# MEGPypes Run Workflow Tutorial

This notebook shows how to interact with the high-level runner endpoint in `src/megpypes/runner.py`.

You will learn how to:
- load and inspect the YAML config (`demos/config_effort.yaml`)
- create a `MegPypesRunner` instance
- build and inspect a workflow before executing
- run the pipeline with default and advanced overrides
- launch the Streamlit QC report app from Python

## 1. Environment Setup

This cell ensures paths resolve correctly whether you run the notebook from the project root or from the demos folder.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "demos":
    os.chdir(cwd.parent)

print(f"Current working directory: {Path.cwd()}")

Current working directory: /Users/peli/Projects/Repositories/MEGPypes


## 2. Load and Inspect the YAML Config

The configuration file is organized into three main sections:

- `paths`: where to read data, write work files, and write outputs
- `workflow`: how to execute (plugin and workers)
- `pipeline_config`: step-level behavior for each pipeline component

In this demo we use `demos/config_effort.yaml`.

In [2]:
import yaml

config_path = Path("demos/config_effort.yaml")
with config_path.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("Top-level keys:", list(config.keys()))
print("paths keys:", list(config["paths"].keys()))
print("workflow keys:", list(config["workflow"].keys()))
print("pipeline_config keys:", list(config["pipeline_config"].keys()))

Top-level keys: ['paths', 'workflow', 'pipeline_config']
paths keys: ['basedir', 'workdir', 'outputdir', 'file_templates', 'iterable_fields', 'iterable_values']
workflow keys: ['plugin', 'n_workers', 'auto_workers']
pipeline_config keys: ['initial_preproc', 'artifact_rejection', 'auto_ica', 'epoching']


## 3. YAML Structure Walkthrough

### paths
- `basedir`: input data root
- `workdir`: Nipype working directory
- `outputdir`: final output location
- `file_templates`: SelectFiles template dictionary
- `iterable_fields` and `iterable_values`: iteration dimensions and values

### workflow
- `plugin`: execution backend (for example `Linear`, `MultiProc`)
- `n_workers`: number of processes when supported
- `auto_workers`: whether to auto-compute worker count

### pipeline_config
Contains per-node step flags and arguments for:
- `initial_preproc`
- `artifact_rejection`
- `auto_ica`
- `epoching`

## 4. Create the Runner

`MegPypesRunner` is the high-level interactive endpoint for this pipeline.

In [3]:
from megpypes.runner import MegPypesRunner

runner = MegPypesRunner.from_yaml(config_path)
print("Runner created from:", runner.config_path)
print("Default plugin:", runner.workflow_config.get("plugin"))

Runner created from: /Users/peli/Projects/Repositories/MEGPypes/demos/config_effort.yaml
Default plugin: Linear


## 5. Build the Workflow (Without Running Yet)

This is useful when you want to inspect or customize before execution.

In [4]:
wf = runner.create_workflow()
print("Workflow name:", wf.name)
print("Workflow base dir:", wf.base_dir)

graph_path = runner.write_graph(wf)
print("Workflow graph path:", graph_path)

raw_dir /Users/peli/Projects/Repositories/MEGPypes/data/effort
Iterating over fields: ['subject', 'session']
Provided iterable values: {'subject': ['0001'], 'session': ['01', '02']}
Processing iterable field: subject
Using custom values for field: subject
Processing iterable field: session
Using custom values for field: session
Final iterables dict: {'subject': ['0001'], 'session': ['01', '02']}
Valid inputs for initial_preproc: {'out_file', 'l_freq', 'trait_added', 'trait_modified', 'gradcomp_order', 'h_freq', 'min_buffer', 'in_file', 'gradcomp_auto', 'stim_channel', 'max_buffer'}
Step 'crop' argument 'stim_channel': None
Step 'crop' argument 'min_buffer': -0.2
Step 'crop' argument 'max_buffer': 0.5
Step 'filter' argument 'l_freq': 1.0
Step 'filter' argument 'h_freq': 100
Step 'gradcomp' argument 'gradcomp_auto': True
Step 'gradcomp' argument 'order': None
Valid inputs for artifact_rejection: {'n_iter_max', 'detect_line_freq', 'trait_added', 'trait_modified', 'out_file', 'ica_h_freq',

## 6. Run the Pipeline (Easy Path)

Use config defaults for plugin and workers.

NiPype caches the stepwise results in the ``workdir`` directory and restarts the workflow from the last saved data. 
Meaningful outputs are stored in the ``output`` directory in a BIDS format.

To completely re-run a workflow from start you can easily delete the "workdir" directory and run again.

Execute when ready (this can take time):

In [5]:
run_info = runner.run(write_graph=True)
print(f"Plugin: {run_info.plugin}")
print(f"Workers: {run_info.n_workers}")
print(f"Graph: {runner.graph_path}")

raw_dir /Users/peli/Projects/Repositories/MEGPypes/data/effort
Iterating over fields: ['subject', 'session']
Provided iterable values: {'subject': ['0001'], 'session': ['01', '02']}
Processing iterable field: subject
Using custom values for field: subject
Processing iterable field: session
Using custom values for field: session
Final iterables dict: {'subject': ['0001'], 'session': ['01', '02']}
260330-18:20:16,330 nipype.workflow DEBUG:
	 (megpreproc.infosource, megpreproc.selectfiles): No edge data
260330-18:20:16,330 nipype.workflow DEBUG:
	 (megpreproc.infosource, megpreproc.selectfiles): new edge data: {'connect': [('subject', 'subject')]}
260330-18:20:16,331 nipype.workflow DEBUG:
	 (megpreproc.infosource, megpreproc.selectfiles): Edge data exists: {'connect': [('subject', 'subject')]}
260330-18:20:16,331 nipype.workflow DEBUG:
	 (megpreproc.infosource, megpreproc.selectfiles): new edge data: {'connect': [('subject', 'subject'), ('session', 'session')]}
Valid inputs for initial_p

/Users/peli/Projects/Repositories/MEGPypes/.venv/lib/python3.13/site-packages/nipype/external/cloghandler.py:145: UserWarning: The given 'filename' should be an absolute path.  If your application calls os.chdir(), your logs may get messed up. Use 'supress_abs_warn=True' to hide this message.
  warn(


260330-18:20:16,564 nipype.workflow INFO:
	 Generated workflow graph: workdir/megpreproc/graph.png (graph2use=colored, simple_form=True).
260330-18:20:16,571 nipype.workflow DEBUG:
	 Creating flat graph for workflow: megpreproc
260330-18:20:16,572 nipype.workflow DEBUG:
	 expanding workflow: megpreproc
260330-18:20:16,573 nipype.workflow DEBUG:
	 processing node: megpreproc.infosource
260330-18:20:16,573 nipype.workflow DEBUG:
	 processing node: megpreproc.selectfiles
260330-18:20:16,573 nipype.workflow DEBUG:
	 processing node: megpreproc.initial_preproc
260330-18:20:16,573 nipype.workflow DEBUG:
	 processing node: megpreproc.artifact_rejection
260330-18:20:16,574 nipype.workflow DEBUG:
	 processing node: megpreproc.datasink
260330-18:20:16,574 nipype.workflow DEBUG:
	 processing node: megpreproc.epoching
260330-18:20:16,574 nipype.workflow DEBUG:
	 processing node: megpreproc.collect_epochs
260330-18:20:16,575 nipype.workflow DEBUG:
	 processing node: megpreproc.build_bids_container


## 7. Advanced Overrides

Override iterable values, then run with explicit execution settings.

In [6]:
# custom_wf = runner.create_workflow(
#    paths_override={
#        "iterable_values": {"subject": ["0001"], "session": ["01"]}
#    }
# )

# print("Custom workflow prepared for one subject/session.")

# advanced_run = runner.run(
#     workflow=custom_wf,
#     plugin="MultiProc",
#     n_workers=4,
# )
# print(advanced_run)

## 8. Launch the Streamlit Report App from Notebook

The runner can start the report app process and point it to a specific output root.

In [ ]:
report_proc = runner.launch_report_app(report_root="workdir/megpreproc/output", port=8501, headless=True)
print("Report URL:", runner.report_url(port=8501))

Report URL: http://127.0.0.1:8501


In [9]:
# To stop later:
runner.stop_report_app()

## 9. Practical Notes

- Keep `demos/config_effort.yaml` under version control as your reproducible baseline.
- Use overrides for one-off exploratory runs.
- If running from JupyterLab on a remote machine, set `host="0.0.0.0"` for report access.
- For large datasets, prefer `MultiProc` with explicit worker limits based on RAM/CPU.